In [1]:
import pandas as pd
import numpy as np

# DataCo dataset requires latin-1 encoding
df = pd.read_csv('../data/DataCoSupplyChainDataset.csv', encoding='latin-1')

print(f"Shape: {df.shape}")
df.head()

Shape: (180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [2]:
df.info()
print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname                 180519 non


Duplicate rows: 0


In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
)
print(df.columns.tolist())

['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_email', 'customer_fname', 'customer_id', 'customer_lname', 'customer_password', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date_dateorders', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'order_zipcode', 'product_card_id', 'product_category_id', 'product_description', 'product_image', 'product_name', 'product_price', 'product_status', 'shipping_date_dateorders', 's

In [4]:
# Drop columns with excessive missingness
threshold = 0.5
missing_pct = df.isnull().mean()
cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
print("Dropping columns:", cols_to_drop)
df.drop(columns=cols_to_drop, inplace=True)

# Fill remaining categorical NaNs
cat_cols = df.select_dtypes(include='str').columns
df[cat_cols] = df[cat_cols].fillna('Unknown')

# Fill numeric NaNs with median
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

print(f"Shape after cleaning: {df.shape}")
print(f"Remaining missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

Dropping columns: ['order_zipcode', 'product_description']


Shape after cleaning: (180519, 51)
Remaining missing values:
Series([], dtype: int64)


In [5]:
# Remove duplicate rows
dup_count = df.duplicated().sum()
print(f"Duplicate rows before removal: {dup_count}")
df.drop_duplicates(inplace=True)
print(f"Shape after deduplication: {df.shape}")

Duplicate rows before removal: 0


Shape after deduplication: (180519, 51)


In [6]:
# Parse date columns to datetime
date_cols = ['order_date_dateorders', 'shipping_date_dateorders']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%m/%d/%Y %H:%M')

print(f"order_date range: {df['order_date_dateorders'].min()} to {df['order_date_dateorders'].max()}")
print(f"shipping_date range: {df['shipping_date_dateorders'].min()} to {df['shipping_date_dateorders'].max()}")

order_date range: 2015-01-01 00:00:00 to 2018-01-31 23:38:00
shipping_date range: 2015-01-03 00:00:00 to 2018-02-06 22:14:00


In [7]:
# Feature engineering
df['order_year'] = df['order_date_dateorders'].dt.year
df['order_month'] = df['order_date_dateorders'].dt.month
df['order_day'] = df['order_date_dateorders'].dt.day
df['order_dayofweek'] = df['order_date_dateorders'].dt.dayofweek
df['order_quarter'] = df['order_date_dateorders'].dt.quarter
df['order_hour'] = df['order_date_dateorders'].dt.hour
df['is_weekend'] = df['order_dayofweek'].isin([5, 6]).astype(int)

df['shipping_year'] = df['shipping_date_dateorders'].dt.year
df['shipping_month'] = df['shipping_date_dateorders'].dt.month

df['actual_shipping_delay'] = df['days_for_shipping_real'] - df['days_for_shipment_scheduled']
df['is_early_or_ontime'] = (df['actual_shipping_delay'] <= 0).astype(int)

df['order_to_shipping_hours'] = (df['shipping_date_dateorders'] - df['order_date_dateorders']).dt.total_seconds() / 3600

season_map = {1: 'Winter', 2: 'Winter', 3: 'Spring', 4: 'Spring', 5: 'Spring',
              6: 'Summer', 7: 'Summer', 8: 'Summer', 9: 'Fall', 10: 'Fall', 11: 'Fall', 12: 'Winter'}
df['order_season'] = df['order_month'].map(season_map)

print(f"Shape after feature engineering: {df.shape}")
print(f"New columns:\n{df.filter(regex='^(order_|shipping_|actual_|is_)').columns.tolist()}")

Shape after feature engineering: (180519, 64)
New columns:
['order_city', 'order_country', 'order_customer_id', 'order_date_dateorders', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'shipping_date_dateorders', 'shipping_mode', 'order_year', 'order_month', 'order_day', 'order_dayofweek', 'order_quarter', 'order_hour', 'is_weekend', 'shipping_year', 'shipping_month', 'actual_shipping_delay', 'is_early_or_ontime', 'order_to_shipping_hours', 'order_season']


In [8]:
# Outlier detection and capping using IQR
outlier_cols = ['benefit_per_order', 'sales_per_customer', 'order_item_discount',
                'order_item_product_price', 'order_item_quantity', 'order_profit_per_order',
                'actual_shipping_delay']

outlier_counts = {}
for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_counts[col] = outliers
    df[col] = df[col].clip(lower, upper)

print("Outliers capped per column:")
for col, count in outlier_counts.items():
    print(f"  {col}: {count} ({count/len(df)*100:.1f}%)")
print(f"\nFinal shape: {df.shape}")

Outliers capped per column:
  benefit_per_order: 18942 (10.5%)
  sales_per_customer: 1943 (1.1%)
  order_item_discount: 7537 (4.2%)
  order_item_product_price: 2048 (1.1%)
  order_item_quantity: 0 (0.0%)
  order_profit_per_order: 18942 (10.5%)
  actual_shipping_delay: 35701 (19.8%)

Final shape: (180519, 64)


In [9]:
# Drop single-value columns (no analytical value)
low_card = [col for col in df.select_dtypes(include='str').columns
            if df[col].nunique() == 1]
if low_card:
    print(f"Dropping single-value columns: {low_card}")
    df.drop(columns=low_card, inplace=True)
    print(f"Shape after dropping low-cardinality columns: {df.shape}")

Dropping single-value columns: ['customer_email', 'customer_password']
Shape after dropping low-cardinality columns: (180519, 62)


In [10]:
# ── Data Audit: Readiness Check ──

print("=" * 60)
print("DATA AUDIT REPORT")
print("=" * 60)

# 1. Missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print("\n\u2713 Missing values: NONE \u2014 clean")
else:
    print(f"\n\u26a0 Missing values:\n{missing}")

# 2. Data types
print(f"\n\u2713 Date types: order_date={df['order_date_dateorders'].dtype}, "
      f"shipping_date={df['shipping_date_dateorders'].dtype}")
print(f"\u2713 Numeric columns: {len(df.select_dtypes(include=np.number).columns)}")
print(f"\u2713 Categorical columns: {len(df.select_dtypes(include='str').columns)}")

# 3. Date sanity
bad_dates = (df['shipping_date_dateorders'] < df['order_date_dateorders']).sum()
print(f"\u2713 Orders shipped before ordered: {bad_dates} (should be 0)")

# 4. Range / sanity checks
print("\n\u2014 Range checks \u2014")
for col in ['order_item_quantity', 'order_item_product_price', 'sales_per_customer']:
    n_neg = (df[col] < 0).sum()
    print(f"  {col}: {n_neg} negative values")

# 5. Low-cardinality columns
low_card = [col for col in df.select_dtypes(include='str').columns
            if df[col].nunique() == 1]
if low_card:
    print(f"\n\u26a0 Columns with single unique value (consider dropping): {low_card}")
else:
    print("\n\u2713 No single-value columns found")

# 6. Key business metrics
print("\n\u2014 Key Metrics \u2014")
late_pct = df['late_delivery_risk'].mean() * 100
print(f"  Late delivery rate: {late_pct:.1f}%")
print(f"  Avg actual shipping delay: {df['actual_shipping_delay'].mean():.2f} days")
print(f"  Unique customers: {df['customer_id'].nunique()}")
print(f"  Unique products: {df['product_card_id'].nunique()}")

# 7. Final summary
print(f"\n{'=' * 60}")
print(f"  Shape: {df.shape}")
print(f"  Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"  Duplicates: {df.duplicated().sum()}")
print(f"{'=' * 60}")
print("STATUS: READY FOR ANALYSIS")

DATA AUDIT REPORT

✓ Missing values: NONE — clean

✓ Date types: order_date=datetime64[us], shipping_date=datetime64[us]
✓ Numeric columns: 39
✓ Categorical columns: 21
✓ Orders shipped before ordered: 0 (should be 0)

— Range checks —
  order_item_quantity: 0 negative values
  order_item_product_price: 0 negative values
  sales_per_customer: 0 negative values

✓ No single-value columns found

— Key Metrics —
  Late delivery rate: 54.8%
  Avg actual shipping delay: 0.55 days
  Unique customers: 20652
  Unique products: 118

  Shape: (180519, 62)
  Memory: 134.2 MB


  Duplicates: 0
STATUS: READY FOR ANALYSIS


In [11]:
# Save cleaned dataset
output_path = '../data/supply_chain_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {df.shape}")

Saved: ../data/supply_chain_cleaned.csv
Shape: (180519, 62)


In [18]:
df.head()

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,order_dayofweek,order_quarter,order_hour,is_weekend,shipping_year,shipping_month,actual_shipping_delay,is_early_or_ontime,order_to_shipping_hours,order_season
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,2,1,22,0,2018,2,-1.0,1,72.0,Winter
1,TRANSFER,5,4,-79.700005,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,5,1,12,1,2018,1,1.0,0,120.0,Winter
2,CASH,4,4,-79.700005,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,5,1,12,1,2018,1,0.0,1,96.0,Winter
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,5,1,11,1,2018,1,-1.0,1,72.0,Winter
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,5,1,11,1,2018,1,-1.5,1,48.0,Winter
